In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
data = {
    "text": [
        "I love machine learning and artificial intelligence",
        "Natural language processing is very interesting",
        "Deep learning models require large datasets",
        "AI is transforming the technology industry",
        "Text preprocessing is important in NLP"
    ],
    "label": [
        "AI",
        "NLP",
        "AI",
        "AI",
        "NLP"
    ]
}

df = pd.DataFrame(data)

print(df)

                                                text label
0  I love machine learning and artificial intelli...    AI
1    Natural language processing is very interesting   NLP
2        Deep learning models require large datasets    AI
3         AI is transforming the technology industry    AI
4             Text preprocessing is important in NLP   NLP


In [4]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["clean_text"] = df["text"].apply(clean_text)

print(df["clean_text"])

0    i love machine learning and artificial intelli...
1      natural language processing is very interesting
2          deep learning models require large datasets
3           ai is transforming the technology industry
4               text preprocessing is important in nlp
Name: clean_text, dtype: object


In [5]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):

    words = text.split()

    filtered_words = [word for word in words if word not in stop_words]

    return " ".join(filtered_words)

df["no_stopwords"] = df["clean_text"].apply(remove_stopwords)

print(df["no_stopwords"])

0    love machine learning artificial intelligence
1          natural language processing interesting
2      deep learning models require large datasets
3              ai transforming technology industry
4                 text preprocessing important nlp
Name: no_stopwords, dtype: object


In [6]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):

    words = text.split()

    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(lemmatized_words)

df["lemmatized_text"] = df["no_stopwords"].apply(lemmatize_text)

print(df["lemmatized_text"])

0    love machine learning artificial intelligence
1          natural language processing interesting
2       deep learning model require large datasets
3              ai transforming technology industry
4                 text preprocessing important nlp
Name: lemmatized_text, dtype: object


In [7]:
label_encoder = LabelEncoder()

df["label_encoded"] = label_encoder.fit_transform(df["label"])

print(df[["label", "label_encoded"]])

  label  label_encoded
0    AI              0
1   NLP              1
2    AI              0
3    AI              0
4   NLP              1


In [8]:
tfidf = TfidfVectorizer()

X_tfidf = tfidf.fit_transform(df["lemmatized_text"])

In [9]:
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

print(tfidf_df)

    ai  artificial  datasets      deep  important  industry  intelligence  \
0  0.0    0.463693  0.000000  0.000000        0.0       0.0      0.463693   
1  0.0    0.000000  0.000000  0.000000        0.0       0.0      0.000000   
2  0.0    0.000000  0.420669  0.420669        0.0       0.0      0.000000   
3  0.5    0.000000  0.000000  0.000000        0.0       0.5      0.000000   
4  0.0    0.000000  0.000000  0.000000        0.5       0.0      0.000000   

   interesting  language     large  ...   machine     model  natural  nlp  \
0          0.0       0.0  0.000000  ...  0.463693  0.000000      0.0  0.0   
1          0.5       0.5  0.000000  ...  0.000000  0.000000      0.5  0.0   
2          0.0       0.0  0.420669  ...  0.000000  0.420669      0.0  0.0   
3          0.0       0.0  0.000000  ...  0.000000  0.000000      0.0  0.0   
4          0.0       0.0  0.000000  ...  0.000000  0.000000      0.0  0.5   

   preprocessing  processing   require  technology  text  transforming  
0

In [10]:
final_df = pd.concat([tfidf_df, df["label_encoded"]], axis=1)

print(final_df)

    ai  artificial  datasets      deep  important  industry  intelligence  \
0  0.0    0.463693  0.000000  0.000000        0.0       0.0      0.463693   
1  0.0    0.000000  0.000000  0.000000        0.0       0.0      0.000000   
2  0.0    0.000000  0.420669  0.420669        0.0       0.0      0.000000   
3  0.5    0.000000  0.000000  0.000000        0.0       0.5      0.000000   
4  0.0    0.000000  0.000000  0.000000        0.5       0.0      0.000000   

   interesting  language     large  ...     model  natural  nlp  \
0          0.0       0.0  0.000000  ...  0.000000      0.0  0.0   
1          0.5       0.5  0.000000  ...  0.000000      0.5  0.0   
2          0.0       0.0  0.420669  ...  0.420669      0.0  0.0   
3          0.0       0.0  0.000000  ...  0.000000      0.0  0.0   
4          0.0       0.0  0.000000  ...  0.000000      0.0  0.5   

   preprocessing  processing   require  technology  text  transforming  \
0            0.0         0.0  0.000000         0.0   0.0    

In [11]:
df.to_csv("processed_text_dataset.csv", index=False)

In [12]:
tfidf_df.to_csv("tfidf_features.csv", index=False)

In [13]:
final_df.to_csv("final_ml_dataset.csv", index=False)